# Bike Rental Prediction: From Linear Baseline to Advanced ML

## Introduction
In Week 2, we finalized our data engineering pipeline using Dagster. This process yielded a clean, unified dataset combining our hourly bike rentals (both direct and registered) with the corresponding historical weather conditions. 

Before we introduce advanced machine learning techniques or complex time-series features, we will start by loading our base dataset to establish a performance baseline.

In [3]:
import pandas as pd

# 1. Work with the old dataset (united_df)
# Loading the CSV exported from our Dagster pipeline
df_united = pd.read_csv('../bikes_dagster/data_warehouse/dfs_united.csv')

# Crucial Fix: Convert the text string back into a mathematical datetime object
df_united['datetime'] = pd.to_datetime(df_united['datetime'])

# 0. Print the head of the dataset to verify our starting point
df_united.head()

,datetime,direct_count,is_weekend,registered_count,total_rentals,is_holiday,conditions,temperature_c,perceived_temperature_c,humidity,windspeed_kmh
0,2011-01-01 00:00:00,3.0,True,13.0,16.0,False,1,3.3,3.0,81.0,0.0
1,2011-01-01 01:00:00,8.0,True,32.0,40.0,False,1,2.3,2.0,80.0,0.0
2,2011-01-01 02:00:00,5.0,True,27.0,32.0,False,1,2.3,2.0,80.0,0.0
3,2011-01-01 03:00:00,3.0,True,10.0,13.0,False,1,3.3,3.0,75.0,0.0
4,2011-01-01 04:00:00,0.0,True,1.0,1.0,False,1,3.3,3.0,75.0,0.0


## Step 2 & 3: The Linear Baseline Model and Initial Evaluation

To understand the true value of advanced machine learning and feature engineering, we must first establish a mathematical baseline. We will train a standard `LinearRegression` model using only the current weather and basic calendar features. 

Because linear algebra draws straight lines through our data, it is highly sensitive to mismatched number scales (e.g., mixing a temperature of `15.5` with a humidity of `80`). To prevent the math engine from failing, we must apply a `StandardScaler` to all continuous features before training.

Most importantly, we set `shuffle=False` during our data split. This guarantees we are strictly training on the past to predict the future, completely eliminating "time-machine" data leaks.

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

In [5]:
# 1. Prepare the DataFrame
model_df = df_united.copy()

1) Convert boolean switches
2) Dismantle datetime
3) Scale continuous features

In [6]:
boolean_features = ['is_weekend', 'is_holiday'] 
model_df[boolean_features] = model_df[boolean_features].astype(int)

model_df['hour'] = model_df['datetime'].dt.hour
model_df['month'] = model_df['datetime'].dt.month

continuous_features = ['hour', 'month', 'temperature_c', 'humidity', 'windspeed_kmh', 'conditions', 'perceived_temperature_c']
scaler = StandardScaler()
model_df[continuous_features] = scaler.fit_transform(model_df[continuous_features])

4) Define X (features) and y (target)
5) Split chronologically

In [7]:
columns_to_drop = ['datetime', 'direct_count', 'registered_count', 'total_rentals']
X = model_df.drop(columns=columns_to_drop)
y = model_df['total_rentals']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

6) Train the mathematical baseline

In [8]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](9,)","[ -0.36,-22.65, -2.96,..., 2.79, 48.03, -1.61]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](9,)","['is_weekend','is_holiday','conditions',...,'windspeed_kmh','hour','month']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,173.3
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,9
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,9


7) Evaluate everything

In [9]:
predictions = linear_model.predict(X_test)
rmse = root_mean_squared_error(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("=== Full Linear Model Evaluation ===")
print(f"RMSE: {rmse:.2f} bikes")
print(f"MAE:  {mae:.2f} bikes")
print(f"R²:   {r2:.2f}")

print("\nScrambling columns to calculate exact feature reliance...")
result = permutation_importance(
    linear_model, 
    X_test, 
    y_test, 
    scoring='neg_root_mean_squared_error', 
    n_repeats=10, 
    random_state=42
)

importance_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance_Score': result.importances_mean
}).sort_values(by='Importance_Score', ascending=False).reset_index(drop=True)

print("\n=== Feature Importance Ranking ===")
print(importance_df)

=== Full Linear Model Evaluation ===
RMSE: 201.08 bikes
MAE:  135.58 bikes
R²:   0.17

Scrambling columns to calculate exact feature reliance...

=== Feature Importance Ranking ===
                   Feature  Importance_Score
0                     hour         16.809232
1  perceived_temperature_c         12.158776
2                 humidity          9.868867
3            temperature_c          2.673032
4            windspeed_kmh          0.236909
5               conditions          0.150650
6               is_holiday          0.143207
7                    month          0.015525
8               is_weekend          0.006024


## Step 4: Analyzing the Baseline and Engineering Memory Features

The evaluation above reveals a critical flaw in our initial approach. An $R^2$ of 0.17 means our baseline model only understands 17% of the real-world renting behavior. Its predictions are off by an average of 201 bikes per hour. 

Looking at the Feature Importance table, the model relies almost entirely on the `hour` of the day and the `perceived_temperature_c`. However, it is completely blind to real-time momentum. It doesn't know if the city is currently experiencing a massive surge in rentals or a sudden drop.

To fix this, we will engineer two new historical time-series features to give our dataset a short-term and long-term memory:
1. **24-Hour Lag (`rentals_24h_ago`):** Acts as a daily anchor, telling the model exactly how busy this specific hour was yesterday.
2. **3-Hour Rolling Average (`rolling_avg_3h`):** Acts as a real-time momentum indicator to capture sudden shifts in demand that the standard weather or calendar features miss.

In [10]:
def engineer_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df_engineered = df.copy()
    
    # Sort strictly by time to ensure shifts work correctly
    df_engineered = df_engineered.sort_values('datetime').reset_index(drop=True)
    
    # 1. Lag Feature: Shift the answers down by exactly 24 rows
    df_engineered['rentals_24h_ago'] = df_engineered['total_rentals'].shift(24)
    
    # 2. Rolling Feature: Calculate the mean of the 3 strictly prior hours
    df_engineered['rolling_avg_3h'] = df_engineered['total_rentals'].shift(1).rolling(window=3).mean()
    
    # Drop the empty rows created by the 24-hour shift
    df_engineered = df_engineered.dropna().reset_index(drop=True)
    
    return df_engineered

In [11]:
df_engineered = engineer_time_features(df_united)

print(df_engineered[['datetime', 'total_rentals', 'rentals_24h_ago', 'rolling_avg_3h']].head())

             datetime  total_rentals  rentals_24h_ago  rolling_avg_3h
0 2011-01-02 00:00:00           17.0             16.0       33.666667
1 2011-01-02 01:00:00           17.0             40.0       28.000000
2 2011-01-02 02:00:00            9.0             32.0       24.333333
3 2011-01-02 03:00:00            6.0             13.0       14.333333
4 2011-01-02 04:00:00            3.0              1.0       10.666667


## Step 5: model re-training:

Now, in order to see the effect of our new feature we reapply our linear_regression model, and evaluate the results.

In [12]:
import model_helpers as mh

In [13]:
mh.linear_model_pipeline(df_engineered)

=== Full Linear Model Evaluation ===
RMSE: 116.32 bikes
MAE:  77.42 bikes
R²:   0.72

Scrambling columns to calculate exact feature reliance...

=== Feature Importance Ranking ===
                    Feature  Importance_Score
0           rentals_24h_ago        116.215906
1            rolling_avg_3h         27.126133
2                  humidity          1.284016
3                conditions          1.108512
4                is_weekend          0.320403
5                      hour          0.294587
6             temperature_c          0.167431
7   perceived_temperature_c          0.103636
8                is_holiday          0.033758
9                     month         -0.000600
10            windspeed_kmh         -0.014982


### Observation: Advanced Linear Regression

As soon as we added history features to our linear model they affect the variables importancy so much that 'hour' column switch to the 5th place dropping from 16.809232 -> 0.294587 in its importance_score.

Additionally, the RMSE imporved and now is off by 116.32 bikes on avarage, comparet to previous 201.08. Which proves that new, feature-engineered columns, are highly efficient.

## Step 6: The Advanced Tree Model

With our dataset now equipped with historical memory, we can upgrade our machine learning engine. We are switching from a basic `LinearRegression` to a `HistGradientBoostingRegressor`. 

This algorithm is mathematically superior for this task for several reasons:
1. **Non-Linearity:** It builds hundreds of decision trees, allowing it to learn complex rules (e.g., "If it is 8:00 AM *and* the rolling average is high *and* it is a weekday -> expect a massive spike").
2. **Immunity to Scaling:** Decision trees only care about the *order* of numbers (e.g., is Temperature > 15?), meaning we can completely drop the `StandardScaler` step.

In [14]:
from sklearn.ensemble import HistGradientBoostingRegressor

In [18]:
model_df = df_engineered.copy()

# Convert boolean switches
boolean_features = ['is_weekend', 'is_holiday'] 
model_df[boolean_features] = model_df[boolean_features].astype(int)

# Dismantle datetime
model_df['hour'] = model_df['datetime'].dt.hour
model_df['month'] = model_df['datetime'].dt.month

# 2. Define X and y (Notice: No scaling needed!)
columns_to_drop = ['datetime', 'direct_count', 'registered_count', 'total_rentals']
X = model_df.drop(columns=columns_to_drop)
y = model_df['total_rentals']

# 3. Split chronologically
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 4. Train the Advanced Tree Model
# max_iter=1000 allows it to build up to 1000 sequential trees
tree_model = HistGradientBoostingRegressor(max_iter=1000, random_state=42)
tree_model.fit(X_train, y_train)

# 5. Evaluate the model
predictions = tree_model.predict(X_test)
rmse = root_mean_squared_error(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("=== Advanced Tree Model Evaluation ===")
print(f"RMSE: {rmse:.2f} bikes")
print(f"MAE:  {mae:.2f} bikes")
print(f"R²:   {r2:.2f}")

=== Advanced Tree Model Evaluation ===
RMSE: 60.98 bikes
MAE:  38.44 bikes
R²:   0.92


In [19]:
# 6. Extract Feature Importance
print("\nScrambling columns to calculate exact feature reliance...")
result = permutation_importance(
    tree_model, 
    X_test, 
    y_test, 
    scoring='neg_root_mean_squared_error', 
    n_repeats=10, 
    random_state=42
)
importance_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance_Score': result.importances_mean
}).sort_values(by='Importance_Score', ascending=False).reset_index(drop=True)

print("\n=== Feature Importance Ranking ===")
print(importance_df)


Scrambling columns to calculate exact feature reliance...

=== Feature Importance Ranking ===
                    Feature  Importance_Score
0            rolling_avg_3h        134.050032
1                      hour         93.985424
2           rentals_24h_ago         62.580810
3                is_weekend         31.268454
4                conditions          4.056217
5   perceived_temperature_c          3.191969
6             temperature_c          3.027033
7                  humidity          2.946387
8                is_holiday          1.553440
9                     month          0.367635
10            windspeed_kmh          0.328083


### Observation: How the Advanced Model "Thinks" Differently

While the engineered memory features drastically improved both models, the **HistGradientBoostingRegressor** processes them in a completely different way than the linear baseline.

Notice the shift in the Feature Importance ranking:
1. **The Comeback of the Clock:** In our advanced linear model, the `hour` feature became mathematically useless (dropping to a score of 0.29). However, the tree model ranks `hour` as its second most critical feature (93.98). 
2. **Momentum over Yesterday:** The tree model heavily prioritizes the 3-hour momentum (`rolling_avg_3h` at 134.05) over yesterday's exact number (`rentals_24h_ago` at 62.58).

**Why does this happen?**
Linear regression only looks for straight-line correlations. It lazily relied on "yesterday's bikes" because the numbers matched up cleanly. 

Decision trees, however, use complex "If-Then" conditional logic. The tree is smart enough to know that a high 3-hour rolling average at 4:00 AM means something completely different than a high rolling average at 8:00 AM. Therefore, the algorithm *must* heavily rely on the `hour` feature to give critical context to the momentum features.

## Step 7: Conclusion

This workflow demonstrates that the secret to high-performance machine learning is not just throwing a complex algorithm at raw data. It is about providing the right mathematical context.

* **The Baseline Failure:** Our initial `LinearRegression` achieved a poor R² of 0.17. It was blind to real-time momentum, resulting in predictions that were off by an average of 201 bikes.
* **The Power of Memory:** By engineering the `rolling_avg_3h` and `rentals_24h_ago` features, we gave the dataset a short-term and long-term memory. Even the rigid linear model improved its RMSE drastically (down to 116 bikes) just by having access to this history.
* **The Algorithmic Leap:** The `HistGradientBoostingRegressor` was the final piece of the puzzle. Unlike the linear model, which lazily relied on yesterday's exact numbers, the tree model intelligently combined the 3-hour momentum with the physical clock to understand *context*. This synergy slashed our error rate and pushed our R² score into the 0.90+ range.

**Next Steps for Production:**
Experimenting in a Jupyter Notebook is perfect for discovery and feature engineering. However, notebooks are fragile. To make this pipeline resilient, automated, and ready for the real world, we have migrated this exact logic into **Dagster Assets**. By doing so, we guarantee that our data merging, feature engineering, and model training happen in a strict, reproducible, and monitored sequence.